# 🧠 NeuroVR — Brain Tumor Segmentation Training

**B.Tech CSE Major Project — AI-Based Brain Tumor Segmentation and 3D Visualization**

> ⚠️ **DISCLAIMER**: This is a research/educational prototype. All outputs are for academic use only and are **NOT** intended for clinical diagnosis.

## This notebook trains:
1. **2D U-Net** — slice-level segmentation on axial BraTS slices
2. **3D SegResNet** — volumetric segmentation with sliding-window inference

## Workflow
```
1. Install dependencies
2. Connect Kaggle & download BraTS 2021
3. Validate dataset (structure + affines + containment)
4. Clone NeuroVR repository
5. Train 2D U-Net  (30 epochs)
6. Train 3D SegResNet (50 epochs)
7. Download checkpoints + metrics
```

**Runtime**: Select `Runtime → Change runtime type → T4 GPU`

## 0️⃣ Check GPU

In [3]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('⚠️  No GPU detected. Go to Runtime → Change runtime type → GPU (T4)')
    print('Training will still work on CPU but will be much slower.')

FileNotFoundError: [Errno 2] No such file or directory: 'nvidia-smi'

## 1️⃣ Install Dependencies

In [1]:
%%capture
!pip install -q \
    monai==1.3.2 \
    nibabel>=5.0.0 \
    scikit-image>=0.20.0 \
    trimesh>=4.0.0 \
    scipy>=1.10.0 \
    pandas>=2.0.0 \
    matplotlib>=3.7.0 \
    kagglehub \
    pyyaml

print('✅ All dependencies installed')

In [2]:
import torch
import monai
import nibabel
print(f'PyTorch  : {torch.__version__}')
print(f'MONAI    : {monai.__version__}')
print(f'NiBabel  : {nibabel.__version__}')
print(f'CUDA     : {torch.cuda.is_available()} | Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

ModuleNotFoundError: No module named 'torch'

## 2️⃣ Kaggle Authentication

You need a **Kaggle API key**. Get it from: https://www.kaggle.com/settings → API → Create New Token

This downloads `kaggle.json` to your machine. Upload it in the next cell.

In [3]:
from google.colab import files
import os, json

print('📁 Upload your kaggle.json file (from https://www.kaggle.com/settings → API → Create Token)')
uploaded = files.upload()

# Install kaggle credentials
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'wb') as f:
    f.write(list(uploaded.values())[0])
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('✅ Kaggle credentials installed')

ModuleNotFoundError: No module named 'google'

## 3️⃣ Download BraTS 2021 Dataset

In [4]:
import os
import kagglehub
print("📥 Downloading BraTS 2021 dataset (~12GB) via kagglehub...")
DATASET_DIR = kagglehub.dataset_download("dschettler8845/brats-2021-task1")
print(f"✅ Download complete. Path: {DATASET_DIR}")


OSError: [Errno 30] Read-only file system: '/content'

In [ ]:
import tarfile, os
from pathlib import Path

EXTRACT_DIR = "/content/data/raw/brats"
os.makedirs(EXTRACT_DIR, exist_ok=True)

tar_files = list(Path(DATASET_DIR).rglob("*.tar"))
print(f"Found tar files: {tar_files}")

for tf in tar_files:
    print(f"Extracting {tf.name}...")
    with tarfile.open(tf, "r") as tar:
        tar.extractall(path=EXTRACT_DIR)

patients = [d for d in Path(EXTRACT_DIR).rglob("*") if d.is_dir() and any(d.glob("*.nii*"))]
print(f"\n✅ Extracted {len(patients)} patient directories")
DATASET_DIR = EXTRACT_DIR


In [ ]:
# If dataset is nested in a subdirectory, find the actual patient root
import os
from pathlib import Path

def find_brats_root(base: str) -> str:
    """Find the directory containing patient folders with .nii.gz files."""
    for root, dirs, files in os.walk(base):
        niftis = [f for f in files if f.endswith('.nii.gz')]
        if niftis:
            return str(Path(root).parent)
    return base

BRATS_ROOT = find_brats_root(DATASET_DIR)
os.environ['BRATS_DATASET_PATH'] = BRATS_ROOT
print(f'✅ BraTS root: {BRATS_ROOT}')
patients = [d for d in Path(BRATS_ROOT).iterdir() if d.is_dir() and any(d.glob('*.nii*'))]
print(f'✅ {len(patients)} patient directories found')

## 4️⃣ Clone NeuroVR Repository

In [ ]:
# ── Option A: Clone from GitHub (if you pushed the code) ─────────────────────
# Replace with your actual GitHub repo URL
GITHUB_REPO = 'https://github.com/Saisuman55/NeuroVR.git'  # UPDATE THIS

!git clone {GITHUB_REPO} /content/neurovr
os.chdir('/content/neurovr')
print(f'Working directory: {os.getcwd()}')

import sys
sys.path.insert(0, '/content/neurovr')

In [ ]:
# Alternatively, if not on GitHub yet, upload a zip of your project:
# from google.colab import files
# uploaded = files.upload()  # Upload brain_tumor_project.zip
# !unzip brain_tumor_project.zip -d /content/neurovr
# os.chdir('/content/neurovr')
# sys.path.insert(0, '/content/neurovr')

## 5️⃣ Validate Dataset

In [ ]:
import sys
sys.path.insert(0, '/content/neurovr')
os.environ['BRATS_DATASET_PATH'] = BRATS_ROOT

from data.dataset_manager import BraTSDatasetManager

mgr = BraTSDatasetManager(dataset_root=BRATS_ROOT)
available = mgr.check_availability()
print(f'Dataset available: {available}')

if available:
    valid_patients, errors = mgr.validate_structure()
    print(f'\nValid patients: {len(valid_patients)}')
    if errors:
        print(f'Errors (first 5): {errors[:5]}')
    
    print('\nDataset statistics:')
    mgr.print_statistics(sample_n=5)
    
    fingerprint = mgr.compute_dataset_fingerprint()
    print(f'Dataset fingerprint: {fingerprint}')

## 6️⃣ Train 2D U-Net (30 epochs)

In [ ]:
# Configuration for 2D training
EPOCHS_2D = 30
BATCH_SIZE_2D = 16  # Larger batch since we're on GPU
LR_2D = 1e-3
PATCH_SIZE = 240
OUTPUT_DIR_2D = '/content/results/2d'

os.makedirs(OUTPUT_DIR_2D, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR_2D}')

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, '-m', 'training.segmentation_2d.train',
    f'--data_dir={BRATS_ROOT}',
    f'--epochs={EPOCHS_2D}',
    f'--batch_size={BATCH_SIZE_2D}',
    f'--lr={LR_2D}',
    f'--patch_size={PATCH_SIZE}',
    f'--output_dir={OUTPUT_DIR_2D}',
    '--orientation=axial',
    '--loss=dice_ce',
    '--early_stopping=10',
    '--tumor_ratio=0.6',
    '--seed=42',
    '--num_workers=2',
    '--device=cuda',
    '--hd95',
]

print('🚀 Starting 2D U-Net training...')
print(f'   Epochs: {EPOCHS_2D} | Batch: {BATCH_SIZE_2D} | LR: {LR_2D}')
print(f'   Output: {OUTPUT_DIR_2D}\n')

env = os.environ.copy()
env['PYTHONPATH'] = '/content/neurovr'

process = subprocess.Popen(
    cmd,
    cwd='/content/neurovr',
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end='')

process.wait()
print(f'\n2D training exit code: {process.returncode}')
if process.returncode == 0:
    print('✅ 2D U-Net training complete!')
else:
    print('❌ 2D training failed — check output above')

In [ ]:
# Show 2D results
import json
from pathlib import Path

metrics_path = Path(OUTPUT_DIR_2D) / 'metrics.json'
if metrics_path.exists():
    with open(metrics_path) as f:
        metrics = json.load(f)
    print('\n📊 2D U-Net Results:')
    print(f'  Best epoch  : {metrics["best_epoch"]}')
    print(f'  Mean Dice   : {metrics["best_mean_dice"]:.4f}')
    m = metrics.get('best_val_metrics', {})
    print(f'  WT Dice     : {m.get("wt_dice", "N/A"):.4f}')
    print(f'  TC Dice     : {m.get("tc_dice", "N/A"):.4f}')
    print(f'  ET Dice     : {m.get("et_dice", "N/A"):.4f}')
    if m.get('mean_hd95'):
        print(f'  Mean HD95   : {m["mean_hd95"]:.2f} mm')
    print(f'  Run ID      : {metrics["run_id"]}')
    print(f'  Git hash    : {metrics["git_hash"][:12]}')

In [ ]:
# Show training curves
import matplotlib.pyplot as plt
import pandas as pd

log_path = Path(OUTPUT_DIR_2D) / 'training_log.csv'
if log_path.exists():
    df = pd.read_csv(log_path)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    axes[0].plot(df['epoch'], df['train_loss'], 'b-o', ms=3, label='Train Loss')
    axes[0].set_title('Training Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].grid(True, alpha=0.3)

    for col, label, color in [('wt_dice', 'WT', '#0ea5e9'), ('tc_dice', 'TC', '#ef4444'), ('et_dice', 'ET', '#f59e0b')]:
        if col in df.columns:
            axes[1].plot(df['epoch'], df[col], '-o', ms=3, label=label, color=color)
    axes[1].set_title('Validation Dice Scores')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    axes[1].set_ylim(0, 1)
    axes[1].grid(True, alpha=0.3)

    if 'mean_hd95' in df.columns:
        axes[2].plot(df['epoch'], df['mean_hd95'], 'g-o', ms=3)
        axes[2].set_title('Mean HD95 (mm)')
        axes[2].set_xlabel('Epoch')
        axes[2].grid(True, alpha=0.3)

    plt.suptitle('2D U-Net Training — NeuroVR', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR_2D}/training_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Plot saved')

## 7️⃣ Train 3D SegResNet (50 epochs)

In [ ]:
# Configuration for 3D training
EPOCHS_3D = 50
BATCH_SIZE_3D = 1
LR_3D = 1e-4
PATCH_SIZE_3D = '96,96,96'  # Larger patches with T4 GPU (16GB)
OUTPUT_DIR_3D = '/content/results/3d'

# Optional: use MONAI pretrained weights for fine-tuning
# This initializes from the MONAI BraTS bundle weights and fine-tunes on real data
USE_PRETRAINED = True
PRETRAINED_PATH = '/content/neurovr/models/monai/brats_mri_segmentation/models/model.pt'

os.makedirs(OUTPUT_DIR_3D, exist_ok=True)
pretrained_exists = Path(PRETRAINED_PATH).exists()
print(f'Pretrained weights available: {pretrained_exists}')
print(f'Output dir: {OUTPUT_DIR_3D}')

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, '-m', 'training.segmentation_3d.train',
    f'--data_dir={BRATS_ROOT}',
    f'--epochs={EPOCHS_3D}',
    f'--batch_size={BATCH_SIZE_3D}',
    f'--lr={LR_3D}',
    f'--patch_size={PATCH_SIZE_3D}',
    f'--output_dir={OUTPUT_DIR_3D}',
    '--architecture=segresnet',
    '--loss=dice_ce',
    '--early_stopping=15',
    '--seed=42',
    '--num_workers=2',
    '--device=cuda',
    '--hd95',
    '--sw_roi_size=96,96,96',
    '--sw_overlap=0.5',
]

# Add pretrained weights if available
if USE_PRETRAINED and pretrained_exists:
    cmd.append(f'--pretrained={PRETRAINED_PATH}')
    print('🔧 Fine-tuning from MONAI pretrained weights')
else:
    print('🔧 Training 3D SegResNet from scratch')

print(f'\n🚀 Starting 3D SegResNet training...')
print(f'   Epochs: {EPOCHS_3D} | Batch: {BATCH_SIZE_3D} | LR: {LR_3D}')
print(f'   Patch:  {PATCH_SIZE_3D} | Output: {OUTPUT_DIR_3D}\n')

env = os.environ.copy()
env['PYTHONPATH'] = '/content/neurovr'

process = subprocess.Popen(
    cmd,
    cwd='/content/neurovr',
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end='')

process.wait()
print(f'\n3D training exit code: {process.returncode}')
if process.returncode == 0:
    print('✅ 3D SegResNet training complete!')
else:
    print('❌ 3D training failed — check output above')

In [ ]:
# Show 3D results
import json
from pathlib import Path

metrics_path = Path(OUTPUT_DIR_3D) / 'metrics.json'
if metrics_path.exists():
    with open(metrics_path) as f:
        metrics = json.load(f)
    print('\n📊 3D SegResNet Results:')
    print(f'  Best epoch  : {metrics["best_epoch"]}')
    print(f'  Mean Dice   : {metrics["best_mean_dice"]:.4f}')
    print(f'  WT Dice     : {metrics.get("best_wt_dice", "N/A")}')
    print(f'  TC Dice     : {metrics.get("best_tc_dice", "N/A")}')
    print(f'  ET Dice     : {metrics.get("best_et_dice", "N/A")}')
    if metrics.get('best_mean_hd95'):
        print(f'  Mean HD95   : {metrics["best_mean_hd95"]:.2f} mm')
    print(f'  Dataset FP  : {metrics.get("dataset_fingerprint", "N/A")[:16]}...')
    print(f'  Run ID      : {metrics["run_id"]}')

In [ ]:
# Show 3D training curves
import matplotlib.pyplot as plt
import pandas as pd

log_path = Path(OUTPUT_DIR_3D) / 'training_log.csv'
if log_path.exists():
    df = pd.read_csv(log_path)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    axes[0].plot(df['epoch'], df['train_loss'], 'b-o', ms=3)
    axes[0].set_title('Training Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].grid(True, alpha=0.3)

    for col, label, color in [('wt_dice', 'WT', '#0ea5e9'), ('tc_dice', 'TC', '#ef4444'), ('et_dice', 'ET', '#f59e0b')]:
        if col in df.columns:
            axes[1].plot(df['epoch'], df[col], '-o', ms=3, label=label, color=color)
    axes[1].set_title('Validation Dice Scores')
    axes[1].legend()
    axes[1].set_ylim(0, 1)
    axes[1].grid(True, alpha=0.3)

    if 'mean_hd95' in df.columns:
        axes[2].plot(df['epoch'], df['mean_hd95'], 'g-o', ms=3)
        axes[2].set_title('Mean HD95 (mm)')
        axes[2].set_xlabel('Epoch')
        axes[2].grid(True, alpha=0.3)

    plt.suptitle('3D SegResNet Training — NeuroVR', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR_3D}/training_summary.png', dpi=150, bbox_inches='tight')
    plt.show()

## 8️⃣ Combined Results Summary

In [ ]:
import json
from pathlib import Path

print('=' * 60)
print('  NeuroVR — Final Training Results')
print('  DISCLAIMER: Research prototype — NOT for clinical use')
print('=' * 60)

for pipeline, out_dir in [('2D U-Net', OUTPUT_DIR_2D), ('3D SegResNet', OUTPUT_DIR_3D)]:
    mp = Path(out_dir) / 'metrics.json'
    if mp.exists():
        m = json.load(open(mp))
        vals = m.get('best_val_metrics', {})
        print(f'\n  {pipeline}:')
        print(f'    Best epoch : {m["best_epoch"]}')
        print(f'    Mean Dice  : {m["best_mean_dice"]:.4f}')
        for r in ['wt', 'tc', 'et']:
            d = vals.get(f'{r}_dice', m.get(f'best_{r}_dice', 'N/A'))
            h = vals.get(f'{r}_hd95', m.get(f'best_{r}_hd95'))
            hd_str = f' | HD95: {h:.2f}mm' if h is not None else ''
            print(f'    {r.upper()} Dice    : {d:.4f if isinstance(d, float) else d}{hd_str}')
    else:
        print(f'\n  {pipeline}: training not completed yet')

print(f'\n  Checkpoints: /content/results/2d/checkpoints/best.pt')
print(f'               /content/results/3d/checkpoints/best.pt')
print('=' * 60)

## 9️⃣ Package & Download Results

In [ ]:
import shutil, os
from pathlib import Path

# Create a clean package with checkpoints + metrics
PKG_DIR = '/content/neurovr_trained_models'
os.makedirs(f'{PKG_DIR}/2d/checkpoints', exist_ok=True)
os.makedirs(f'{PKG_DIR}/3d/checkpoints', exist_ok=True)

files_to_copy = [
    # 2D results
    (f'{OUTPUT_DIR_2D}/checkpoints/best.pt',   f'{PKG_DIR}/2d/checkpoints/best.pt'),
    (f'{OUTPUT_DIR_2D}/metrics.json',          f'{PKG_DIR}/2d/metrics.json'),
    (f'{OUTPUT_DIR_2D}/manifest.json',         f'{PKG_DIR}/2d/manifest.json'),
    (f'{OUTPUT_DIR_2D}/training_log.csv',      f'{PKG_DIR}/2d/training_log.csv'),
    (f'{OUTPUT_DIR_2D}/training_summary.png',  f'{PKG_DIR}/2d/training_summary.png'),
    # 3D results
    (f'{OUTPUT_DIR_3D}/checkpoints/best.pt',   f'{PKG_DIR}/3d/checkpoints/best.pt'),
    (f'{OUTPUT_DIR_3D}/metrics.json',          f'{PKG_DIR}/3d/metrics.json'),
    (f'{OUTPUT_DIR_3D}/manifest.json',         f'{PKG_DIR}/3d/manifest.json'),
    (f'{OUTPUT_DIR_3D}/training_log.csv',      f'{PKG_DIR}/3d/training_log.csv'),
    (f'{OUTPUT_DIR_3D}/training_summary.png',  f'{PKG_DIR}/3d/training_summary.png'),
]

for src, dst in files_to_copy:
    if Path(src).exists():
        shutil.copy2(src, dst)
        print(f'✅ Copied: {Path(src).name}')
    else:
        print(f'⚠️  Missing: {src}')

print('\n📦 Creating zip archive...')
shutil.make_archive('/content/neurovr_trained_models', 'zip', PKG_DIR)
print('✅ Package ready: /content/neurovr_trained_models.zip')

In [ ]:
from google.colab import files
print('📥 Downloading trained models package...')
files.download('/content/neurovr_trained_models.zip')
print('Done! Extract and place checkpoints in:')
print('  → results/2d/checkpoints/best.pt')
print('  → results/3d/checkpoints/best.pt')

## 🔟 Integrate with NeuroVR Flask App

After downloading, integrate the checkpoints:

```bash
# On your local machine:
unzip neurovr_trained_models.zip -d neurovr_trained_models/
cp neurovr_trained_models/2d/checkpoints/best.pt results/2d/checkpoints/best.pt
cp neurovr_trained_models/3d/checkpoints/best.pt results/3d/checkpoints/best.pt

# Then run the NeuroVR app
python3 flask_app_3d.py
```

The 2D inference endpoint will automatically use `results/2d/checkpoints/best.pt`  
and the 3D inference endpoint will use `results/3d/checkpoints/best.pt`.

---

## 📋 Academic Reporting Template

Use these results in your final report. Fill in actual values after training:

| Model | Dataset | WT Dice | TC Dice | ET Dice | Mean Dice | WT HD95 | TC HD95 | ET HD95 |
|-------|---------|---------|---------|---------|-----------|---------|---------|----------|
| 2D U-Net | BraTS 2021 | _fill_ | _fill_ | _fill_ | _fill_ | _fill_ mm | _fill_ mm | _fill_ mm |
| 3D SegResNet | BraTS 2021 | _fill_ | _fill_ | _fill_ | _fill_ | _fill_ mm | _fill_ mm | _fill_ mm |

> ⚠️ Always cite: *Baid et al., 'The RSNA-ASNR-MICCAI BraTS 2021 Benchmark', arXiv:2107.02314*
>
> ⚠️ Always include: *'This is a research/educational prototype and NOT a clinical diagnostic system.'*